# Laboratorio 02 — Análisis exploratorio (Nivel B)

Este notebook **solo** importa funciones de `src/` y genera figuras; no define
indicadores, el motor de backtesting, ni la lógica de optimización (esa vive
exclusivamente en `src/`). Tiene dos partes:

1. Un recorrido **ilustrativo y rápido** (auditoría de datos, indicadores,
   una muestra pequeña del walk-forward) para explicar cómo funciona cada
   pieza.
2. La **lectura de los resultados completos y definitivos**, que ya fueron
   calculados por `python main.py` y quedaron guardados en `docs/tablas/` y
   `docs/figuras/`. Los números que importan para el reporte son esos, no
   los de la sección ilustrativa.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src import config, data, signals, backtest, metrics, regimes, optimize, plots

pd.set_option("display.max_columns", 40)

## 1. Auditoría de datos (src.data)

In [2]:
train, test, audit_train, audit_test = data.load_train_test()
print(audit_train.resumen_texto())
print()
print(audit_test.resumen_texto())

Auditoría de btc_project_train.csv
  Filas: 166,450 | Rango: 2022-06-01 00:00:00 -> 2023-12-31 00:00:00
  Gmtoffset único: [0] (0 = UTC, confirmado)
  Barras con OHLC vacío: 3,745 en 272 rachas (racha máxima: 270 barras)
  Marcas fuera de rejilla de 5 min: 573
  Barras planas (O=H=L=C): 3,355
  Volumen vacío: 47.5%
  Huecos grandes de calendario (> 30 min): 2
    - 2023-03-12 01:55:00 -> 2023-03-12 03:00:00 (1.08 h)
    - 2023-03-25 23:55:00 -> 2023-03-28 00:00:00 (48.08 h)

Auditoría de btc_project_test.csv
  Filas: 9,387 | Rango: 2023-12-31 00:00:00 -> 2024-06-03 03:27:00
  Gmtoffset único: [0] (0 = UTC, confirmado)
  Barras con OHLC vacío: 17 en 9 rachas (racha máxima: 4 barras)
  Marcas fuera de rejilla de 5 min: 3
  Barras planas (O=H=L=C): 29
  Volumen vacío: 51.6%
  Huecos grandes de calendario (> 30 min): 1
    - 2023-12-31 23:55:00 -> 2024-05-02 13:30:00 (2941.58 h)


In [3]:
display(data.episode_summary(train))
display(data.episode_summary(test))

,episode_id,inicio,fin,n_barras,n_barras_operables
0,1,2022-06-01 00:00:00,2023-03-12 01:55:00,80502,78486
1,2,2023-03-12 03:00:00,2023-03-25 23:55:00,3711,1695
2,3,2023-03-28 00:00:00,2023-12-31 00:00:00,78492,76476


,episode_id,inicio,fin,n_barras,n_barras_operables
0,1,2023-12-31 00:05:00,2023-12-31 23:55:00,288,0
1,2,2024-05-02 13:30:00,2024-06-03 03:27:00,9081,7065


## 2. Indicadores y regla de confirmación (src.signals)

Se calcula la señal confirmada sobre un tramo corto del episodio más largo
del train, solo para visualizar cómo se ven los tres indicadores y la
confirmación 2 de 3 sobre el precio.

In [4]:
episodio_largo = train[train["episode_id"] == train["episode_id"].value_counts().idxmax()].reset_index(drop=True)
muestra = episodio_largo.iloc[3000:5000].reset_index(drop=True)

parametros_demo = signals.StrategyParams(
    ema_fast=12, ema_slow=96, trend_band_k=0.2,
    rsi_window=14, rsi_band=10.0,
    boll_window=24, boll_k=2.2,
    atr_window=14, sl_atr_mult=2.0, tp_atr_mult=3.0,
    position_fraction=0.3,
)
tabla_demo = signals.compute_indicator_table(muestra, parametros_demo)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(muestra["Datetime"], muestra["Close"], color="black", linewidth=0.8, label="Close")
largos = tabla_demo["signal"] == 1
cortos = tabla_demo["signal"] == -1
ax.scatter(muestra.loc[largos, "Datetime"], muestra.loc[largos, "Close"], color="green", s=12, label="Señal larga (2 de 3)")
ax.scatter(muestra.loc[cortos, "Datetime"], muestra.loc[cortos, "Close"], color="red", s=12, label="Señal corta (2 de 3)")
ax.set_title("Señales de la estrategia sobre una muestra de train")
ax.set_xlabel("Fecha")
ax.set_ylabel("Precio")
ax.legend()
plt.show()

## 3. Muestra rápida del walk-forward (ilustrativa, pocas ventanas)

**Esto NO reemplaza la corrida completa de `python main.py`** (76 ventanas,
~15-25 minutos). Aquí solo se corren unas pocas ventanas para mostrar cómo
funciona la optimización híbrida Random Search + TPE y la detección de
régimen, en un tiempo razonable dentro del notebook.

In [5]:
wf_demo = optimize.run_walkforward(train, seed=config.RANDOM_SEED, max_windows=4, verbose=True)
wf_demo.windows_table[["window", "regime", "n_trials", "selection_method", "n_trades_is", "calmar_is", "n_trades_oos", "calmar_oos", "wfe"]]

  ventana 1/4: 2022-06-01 -> 2022-07-08 | configs acumuladas=360


  ventana 2/4: 2022-06-08 -> 2022-07-15 | configs acumuladas=720


  ventana 3/4: 2022-06-15 -> 2022-07-22 | configs acumuladas=1080


  ventana 4/4: 2022-06-22 -> 2022-07-29 | configs acumuladas=1440


,window,regime,n_trials,selection_method,n_trades_is,calmar_is,n_trades_oos,calmar_oos,wfe
0,0,tendencia,120,meseta,35,83.210268,22,-20.177068,-0.242483
1,0,reversion,120,meseta,430,-1.556778,217,-2.434282,NaN
2,0,crisis,120,meseta,375,-1.649363,0,NaN,NaN
3,1,tendencia,120,meseta,126,-3.253758,22,-16.766314,NaN
4,1,reversion,120,meseta,377,-1.592503,169,-2.596327,NaN
5,1,crisis,120,meseta,238,-2.593087,0,NaN,NaN
6,2,tendencia,120,meseta,471,-1.559368,299,-2.029261,NaN
7,2,reversion,120,meseta,327,-1.842242,0,NaN,NaN
8,2,crisis,120,meseta,433,-1.555368,0,NaN,NaN
9,3,tendencia,120,meseta,643,-1.238682,278,-2.085135,NaN


In [6]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(wf_demo.oos_equity.index, wf_demo.oos_equity.values, color="#1f6feb")
ax.set_title("Curva de equity OOS - muestra de 4 ventanas de walk-forward")
ax.set_xlabel("Fecha")
ax.set_ylabel("Valor del portafolio (USD)")
plt.show()

## 4. Resultados completos (generados por `python main.py`)

A partir de aquí se leen los archivos reales en `docs/tablas/`, producidos
por la corrida completa del walk-forward (76 ventanas) y de la evaluación
en test. Estos son los números que se reportan en `docs/reporte.pdf`.

In [7]:
tablas_dir = config.TABLES_DIR
metricas = pd.read_csv(tablas_dir / "metricas_resumen.csv")
metricas

,Unnamed: 0,conjunto,sharpe,sortino,calmar,max_drawdown,win_rate,retorno_anualizado,volatilidad_anualizada,n_operaciones,payoff_ratio,retorno_total,benchmark_retorno_total,benchmark_calmar,benchmark_max_drawdown
0,0,train_oos_walkforward,-113.312242,-73.048849,-1.000000,1.000000,0.055710,-1.000000,0.277050,18686,1.336013,-1.000000,1.148194,1.760934,0.378899
1,1,test_completo,-117.698761,-64.904893,-1.064122,0.938408,0.053390,-0.998581,0.265389,1180,0.318328,-0.938408,0.618088,26.690272,0.078777
2,2,test_segmento_dic2023,-147.681842,-78.961319,-10.054995,0.099453,0.000000,-1.000000,0.259527,43,NaN,-0.099453,NaN,NaN,NaN
3,3,test_segmento_may2024,-116.787695,-64.465190,-1.073416,0.931606,0.055409,-1.000000,0.265569,1137,0.341629,-0.931606,NaN,NaN,NaN


In [8]:
ventanas = pd.read_csv(tablas_dir / "walkforward_ventanas.csv")
print(f"Ventanas optimizadas: {ventanas['window'].nunique()}")
print(f"WFE media (Calmar_IS > 0.05): {ventanas['wfe'].dropna().mean():.3f}")
ventanas.groupby("regime")[["calmar_is", "calmar_oos", "wfe", "n_trades_is", "n_trades_oos"]].mean()

Ventanas optimizadas: 74
WFE media (Calmar_IS > 0.05): -1.220


,calmar_is,calmar_oos,wfe,n_trades_is,n_trades_oos
regime,,,,,
crisis,79.187778,1.879067e+07,-0.002050,315.675676,68.891892
reversion,-1.934401,-1.459824e+01,NaN,433.743243,80.013514
tendencia,14.859216,-8.846227e+00,-1.524356,346.148649,103.135135


In [9]:
sensibilidad = pd.read_csv(tablas_dir / "sensibilidad_parametros.csv")
sensibilidad.head(15)

,Unnamed: 0,regimen,parametro,variacion,calmar,retorno_total
0,0,base,ninguno,0.0,-1.0,-1.0
1,1,tendencia,ema_fast,+20%,-1.0,-1.0
2,2,tendencia,ema_fast,-20%,-1.0,-1.0
3,3,tendencia,ema_slow,+20%,-1.0,-1.0
4,4,tendencia,ema_slow,-20%,-1.0,-1.0
5,5,tendencia,trend_band_k,+20%,-1.0,-1.0
6,6,tendencia,trend_band_k,-20%,-1.0,-1.0
7,7,tendencia,rsi_window,+20%,-1.0,-1.0
8,8,tendencia,rsi_window,-20%,-1.0,-1.0
9,9,tendencia,rsi_band,+20%,-1.0,-1.0


In [10]:
costos = pd.read_csv(tablas_dir / "sensibilidad_costos.csv")
costos

,Unnamed: 0,comision,retorno_total,calmar,n_operaciones
0,0,0.000000,-0.594724,-0.701236,22521
1,1,0.000625,-1.000000,-1.000000,22521
2,2,0.001250,-1.000000,-1.000000,22521
3,3,0.002500,-1.000000,-1.000000,22521
4,4,0.003750,-1.000000,-1.000000,22521
5,5,0.005000,-1.000000,-1.000000,22521
6,6,0.007222,-1.000000,-1.000000,22521
7,7,0.009444,-1.000000,-1.000000,22521
8,8,0.011667,-1.000000,-1.000000,22521
9,9,0.013889,-1.000000,-1.000000,22521


In [11]:
comparacion = pd.read_csv(tablas_dir / "comparacion_confirmacion_vs_individual.csv")
comparacion

,Unnamed: 0,variante,n_operaciones,calmar,retorno_total
0,0,confirmacion_2_de_3,22521,-1.000000,-1.0
1,1,solo_tendencia,39536,-1.000000,-1.0
2,2,solo_momento,30528,-1.000000,-1.0
3,3,solo_volatilidad,7495,-0.999995,-1.0


In [12]:
regimen_duracion_train = pd.read_csv(tablas_dir / "regimen_duracion_train.csv")
regimen_duracion_test = pd.read_csv(tablas_dir / "regimen_duracion_test.csv")
regimen_proporcion = pd.read_csv(tablas_dir / "regimen_proporcion_tiempo.csv")
display(regimen_duracion_train)
display(regimen_duracion_test)
display(regimen_proporcion)

,Unnamed: 0,regimen,duracion_media_horas,n_episodios,n_transiciones_total,regimen_nombre
0,0,0,80.435606,66,177,tendencia
1,1,1,52.985556,75,177,reversion
2,2,2,79.207207,37,177,crisis


,Unnamed: 0,regimen,duracion_media_horas,n_episodios,n_transiciones_total,regimen_nombre
0,0,0,4.666667,3,9,tendencia
1,1,1,133.687500,4,9,reversion
2,2,2,20.000000,3,9,crisis


,Unnamed: 0,train,test
0,tendencia,0.434668,0.022998
1,reversion,0.325375,0.878439
2,crisis,0.239956,0.098563


In [13]:
with open(tablas_dir / "friccion_ejecucion_test.json", encoding="utf-8") as f:
    friccion = json.load(f)
friccion

{'half_spread_frac': 0.00013092015613807992,
 'sigma_bar': 0.004200112235972216,
 'impacto_promedio_frac': 1.4406539714305471e-05,
 'friccion_total_frac_por_lado': 0.00014532669585238538,
 'pnl_original': -938407.5973229666,
 'pnl_con_friccion': -1050751.574212861,
 'costo_friccion_total': 112343.97688989449,
 'n_operaciones': 1180}

In [14]:
with open(tablas_dir / "parametros_congelados.json", encoding="utf-8") as f:
    congelado = json.load(f)
congelado["params_by_regime"]

{'tendencia': {'ema_fast': 5,
  'ema_slow': 188,
  'trend_band_k': 0.9387554773567128,
  'rsi_window': 19,
  'rsi_band': 6.704490230265973,
  'boll_window': 46,
  'boll_k': 3.392567390057672,
  'atr_window': 16,
  'sl_atr_mult': 1.0091712253024836,
  'tp_atr_mult': 1.038432627134814,
  'position_fraction': 0.909213455136407},
 'reversion': {'ema_fast': 27,
  'ema_slow': 61,
  'trend_band_k': 0.03280965568820392,
  'rsi_window': 7,
  'rsi_band': 7.235251300278558,
  'boll_window': 40,
  'boll_k': 1.894753954677494,
  'atr_window': 29,
  'sl_atr_mult': 1.0773672902538314,
  'tp_atr_mult': 1.4024160040824518,
  'position_fraction': 0.934752524804874},
 'crisis': {'ema_fast': 17,
  'ema_slow': 57,
  'trend_band_k': 0.6382326935477489,
  'rsi_window': 16,
  'rsi_band': 6.610113161895492,
  'boll_window': 55,
  'boll_k': 1.6214008608279253,
  'atr_window': 16,
  'sl_atr_mult': 3.621096734874745,
  'tp_atr_mult': 1.158935570953226,
  'position_fraction': 0.9564107210553285}}

## 5. Figuras

Todas las figuras obligatorias y de diagnóstico ya están generadas en
`docs/figuras/` por `python main.py`. Se listan a continuación.

In [15]:
for p in sorted(config.FIGURES_DIR.glob("*.png")):
    print(p.name)

fig01a_equity_train.png
fig01b_equity_test.png
fig02a_drawdown_train.png
fig02b_drawdown_test.png
fig03a_retornos_mensuales_train.png
fig03b_retornos_mensuales_test.png
fig04_sensibilidad_parametros.png
fig05_retorno_vs_costo.png
fig06a_regimen_timeline_train.png
fig06b_distribuciones_regimen.png
fig06c_equity_regimenes.png
fig07_historia_crisis.png
fig07_historia_reversion.png
fig07_historia_tendencia.png
fig08_importancia_crisis.png
fig08_importancia_reversion.png
fig08_importancia_tendencia.png
fig09_slice_crisis.png
fig09_slice_reversion.png
fig09_slice_tendencia.png
fig10_superficie_crisis.png
fig10_superficie_reversion.png
fig10_superficie_tendencia.png
